# Notebook 125: Seq2Seq ― シーケンス変換
## Sequence-to-Sequence: Encoder-Decoder Architecture
---
### このノートブックの位置づけ
**Unit 8「シーケンスモデリング」** の応用編。ここからPyTorchに移行し、可変長入出力を扱うEncoder-Decoderアーキテクチャを学びます。

### 学習目標
1. **Encoder-Decoderアーキテクチャ** の構造と情報フローを理解する
2. **コンテキストベクトル** の役割とボトルネックを実証する
3. **Teacher Forcing** の仕組みと効果を実装・比較できる
4. **PyTorchでSeq2Seq** を実装し訓練できる

### 前提知識
- Notebook 121-124（RNN, LSTM, GRU）
- PyTorchの基礎（テンソル操作、nn.Module、autograd）

⏱️ **推定学習時間**: 150-180分  
📊 **難易度**: ★★★★☆（上級）  
🎓 **カテゴリ**: 応用

## 📋 目次

1. [なぜSeq2Seqが必要か](#1.-なぜSeq2Seqが必要か)
2. [Encoder-Decoderアーキテクチャ](#2.-Encoder-Decoderアーキテクチャ)
3. [PyTorchでのSeq2Seq実装](#3.-PyTorchでのSeq2Seq実装)
4. [数値系列逆順タスク](#4.-数値系列逆順タスク)
5. [Teacher Forcing](#5.-Teacher-Forcing)
6. [コンテキストベクトルのボトルネック](#6.-コンテキストベクトルのボトルネック)
7. [まとめ](#7.-まとめ)
8. [自己評価クイズ](#8.-自己評価クイズ)

In [ ]:
# ============================================================
# 環境セットアップ
# 必要なライブラリをインポートし、再現性を確保します
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定
plt.rcParams['font.family'] = ['Hiragino Sans', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# 再現性のための乱数シード固定
np.random.seed(42)
torch.manual_seed(42)

# デバイス設定
device = torch.device('cpu')
print(f"PyTorch version: {torch.__version__}")
print("環境セットアップ完了")

# ============================================================
# 1. なぜSeq2Seqが必要か
# ============================================================

## 1. なぜSeq2Seqが必要か

### 🤔 従来モデルの限界

これまで学んだRNN/LSTM/GRUは、以下のタスクに使用できました：

- **Many-to-One**: 系列分類（感情分析など）— 可変長入力 → 固定出力
- **One-to-Many**: 系列生成（画像キャプション）— 固定入力 → 可変長出力
- **Many-to-Many（同じ長さ）**: 品詞タグ付け — 入力と出力が同じ長さ

しかし、現実の多くの問題は **入力と出力の長さが異なります**：

| タスク | 入力 | 出力 | 長さの関係 |
|--------|------|------|------------|
| 機械翻訳 | "I love cats" (3語) | "私は猫が好きです" (5語) | 異なる |
| 文書要約 | 長い文章 (100語) | 短い要約 (20語) | 異なる |
| チャットボット | 質問 (可変) | 回答 (可変) | 異なる |
| 音声認識 | 音声フレーム (1000+) | テキスト (数十語) | 大きく異なる |

### 💡 解決策: Sequence-to-Sequence (Seq2Seq)

**Seq2Seq** は、可変長の入力系列を受け取り、可変長の出力系列を生成するアーキテクチャです。

```
入力系列（任意の長さ）→ [Encoder] → コンテキストベクトル → [Decoder] → 出力系列（任意の長さ）
```

2014年にSutskeverら（Google）とChoら（モントリオール大学）がほぼ同時に提案し、
機械翻訳の性能を大幅に向上させました。

In [ ]:
# ============================================================
# 固定長 vs 可変長の問題を視覚化
# 従来のRNNとSeq2Seqの入出力の違いを図示します
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左図: 従来のRNN (Many-to-One) ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_title('従来のRNN: 固定長出力', fontsize=14, fontweight='bold')
ax.axis('off')

# 入力ボックス
input_words = ['I', 'love', 'cats']
for i, word in enumerate(input_words):
    rect = mpatches.FancyBboxPatch((1 + i * 2.5, 3.5), 1.8, 1.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#3498db', alpha=0.8)
    ax.add_patch(rect)
    ax.text(1.9 + i * 2.5, 4.0, word, ha='center', va='center',
            fontsize=12, color='white', fontweight='bold')

# 矢印
ax.annotate('', xy=(5, 3.2), xytext=(5, 2.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# 出力ボックス（固定サイズ1つ）
rect = mpatches.FancyBboxPatch((3.5, 1.0), 3.0, 1.0,
                                boxstyle='round,pad=0.1',
                                facecolor='#e74c3c', alpha=0.8)
ax.add_patch(rect)
ax.text(5, 1.5, 'Positive 😊', ha='center', va='center',
        fontsize=12, color='white', fontweight='bold')

ax.text(5, 5.2, '入力: 3トークン → 出力: 1ラベル', ha='center',
        fontsize=10, style='italic', color='gray')

# --- 右図: Seq2Seq (可変長→可変長) ---
ax = axes[1]
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.set_title('Seq2Seq: 可変長入出力', fontsize=14, fontweight='bold')
ax.axis('off')

# 入力ボックス
input_words = ['I', 'love', 'cats']
for i, word in enumerate(input_words):
    rect = mpatches.FancyBboxPatch((0.5 + i * 2.0, 3.5), 1.5, 1.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#3498db', alpha=0.8)
    ax.add_patch(rect)
    ax.text(1.25 + i * 2.0, 4.0, word, ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')

# 矢印
ax.annotate('', xy=(6, 3.2), xytext=(6, 2.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# 出力ボックス（可変長: 5トークン）
output_words = ['私は', '猫が', '好き', 'です', '<EOS>']
for i, word in enumerate(output_words):
    rect = mpatches.FancyBboxPatch((0.5 + i * 2.2, 1.0), 1.7, 1.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#2ecc71', alpha=0.8)
    ax.add_patch(rect)
    ax.text(1.35 + i * 2.2, 1.5, word, ha='center', va='center',
            fontsize=10, color='white', fontweight='bold')

ax.text(6, 5.2, '入力: 3トークン → 出力: 5トークン', ha='center',
        fontsize=10, style='italic', color='gray')

plt.tight_layout()
plt.show()

print("Seq2Seqは入力と出力の長さが異なるタスクを扱えます")

# ============================================================
# 2. Encoder-Decoderアーキテクチャ
# ============================================================

## 2. Encoder-Decoderアーキテクチャ

### 📊 全体構造

Seq2Seqは **Encoder（符号化器）** と **Decoder（復号化器）** の2つのRNNから構成されます。

```
【Encoder】                              【Decoder】

  x₁    x₂    x₃   <EOS>                <SOS>   y₁    y₂    y₃
  │     │     │     │                    │     │     │     │
  ▼     ▼     ▼     ▼                    ▼     ▼     ▼     ▼
┌───┐ ┌───┐ ┌───┐ ┌───┐    context    ┌───┐ ┌───┐ ┌───┐ ┌───┐
│ h₁│→│ h₂│→│ h₃│→│ h₄│──────────────→│ s₁│→│ s₂│→│ s₃│→│ s₄│
└───┘ └───┘ └───┘ └───┘    vector      └───┘ └───┘ └───┘ └───┘
                                          │     │     │     │
                                          ▼     ▼     ▼     ▼
                                          ŷ₁    ŷ₂    ŷ₃   <EOS>
```

### 🔑 重要な概念

#### Encoder（符号化器）
- 入力系列を1トークンずつ読み込む
- 最終隠れ状態が **コンテキストベクトル（Context Vector）** になる
- 入力系列全体の情報を固定長ベクトルに圧縮する役割

#### コンテキストベクトル
- Encoderの最終隠れ状態 `h_T`
- 入力系列の「意味」を固定長ベクトルに凝縮したもの
- Decoderの初期状態として渡される
- **ボトルネック**: すべての情報がこの1つのベクトルを通過する

#### Decoder（復号化器）
- コンテキストベクトルを初期状態として受け取る
- `<SOS>`（Start of Sequence）トークンから生成を開始
- 各ステップで1トークンを予測し、次の入力にする
- `<EOS>`（End of Sequence）を予測したら終了

In [ ]:
# ============================================================
# Encoder-Decoderアーキテクチャの詳細図
# 情報の流れを視覚的に表現します
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Encoder-Decoder アーキテクチャの情報フロー', fontsize=16, fontweight='bold')

# 色の定義
enc_color = '#3498db'   # Encoder: 青
dec_color = '#2ecc71'   # Decoder: 緑
ctx_color = '#e74c3c'   # Context: 赤
emb_color = '#9b59b6'   # Embedding: 紫
out_color = '#f39c12'   # Output: オレンジ

# --- Encoder側 ---
ax.text(3.5, 9.2, 'Encoder (LSTM)', ha='center', fontsize=14,
        fontweight='bold', color=enc_color)

# 入力トークン
enc_tokens = ['x₁', 'x₂', 'x₃', 'EOS']
for i, tok in enumerate(enc_tokens):
    x_pos = 1 + i * 1.8
    # 入力
    ax.text(x_pos, 1.5, tok, ha='center', va='center', fontsize=12,
            fontweight='bold', color='#333')
    # Embedding
    rect = mpatches.FancyBboxPatch((x_pos - 0.6, 2.5), 1.2, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor=emb_color, alpha=0.7)
    ax.add_patch(rect)
    ax.text(x_pos, 2.9, 'Emb', ha='center', va='center', fontsize=9, color='white')
    # 矢印: 入力→Embedding
    ax.annotate('', xy=(x_pos, 2.5), xytext=(x_pos, 1.8),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
    # LSTM セル
    rect = mpatches.FancyBboxPatch((x_pos - 0.6, 4.2), 1.2, 1.2,
                                    boxstyle='round,pad=0.1',
                                    facecolor=enc_color, alpha=0.8)
    ax.add_patch(rect)
    ax.text(x_pos, 4.8, f'h{i+1}', ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    # 矢印: Embedding→LSTM
    ax.annotate('', xy=(x_pos, 4.2), xytext=(x_pos, 3.3),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
    # 矢印: LSTM→LSTM (横)
    if i < len(enc_tokens) - 1:
        ax.annotate('', xy=(x_pos + 1.2, 4.8), xytext=(x_pos + 0.6, 4.8),
                    arrowprops=dict(arrowstyle='->', lw=2, color=enc_color))

# --- コンテキストベクトル ---
ctx_x, ctx_y = 8.0, 4.8
rect = mpatches.FancyBboxPatch((ctx_x - 0.8, ctx_y - 0.6), 1.6, 1.2,
                                boxstyle='round,pad=0.1',
                                facecolor=ctx_color, alpha=0.9)
ax.add_patch(rect)
ax.text(ctx_x, ctx_y, 'Context\nVector', ha='center', va='center',
        fontsize=10, color='white', fontweight='bold')

# 矢印: 最後のEncoder → コンテキスト
ax.annotate('', xy=(ctx_x - 0.8, ctx_y), xytext=(6.8, ctx_y),
            arrowprops=dict(arrowstyle='->', lw=2.5, color=ctx_color))

# --- Decoder側 ---
ax.text(12.5, 9.2, 'Decoder (LSTM)', ha='center', fontsize=14,
        fontweight='bold', color=dec_color)

dec_inputs = ['SOS', 'ŷ₁', 'ŷ₂', 'ŷ₃']
dec_outputs = ['ŷ₁', 'ŷ₂', 'ŷ₃', 'EOS']

for i, (inp, out) in enumerate(zip(dec_inputs, dec_outputs)):
    x_pos = 9.5 + i * 1.8
    # 入力トークン
    ax.text(x_pos, 1.5, inp, ha='center', va='center', fontsize=12,
            fontweight='bold', color='#333')
    # Embedding
    rect = mpatches.FancyBboxPatch((x_pos - 0.6, 2.5), 1.2, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor=emb_color, alpha=0.7)
    ax.add_patch(rect)
    ax.text(x_pos, 2.9, 'Emb', ha='center', va='center', fontsize=9, color='white')
    # 矢印: 入力→Embedding
    ax.annotate('', xy=(x_pos, 2.5), xytext=(x_pos, 1.8),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
    # LSTM セル
    rect = mpatches.FancyBboxPatch((x_pos - 0.6, 4.2), 1.2, 1.2,
                                    boxstyle='round,pad=0.1',
                                    facecolor=dec_color, alpha=0.8)
    ax.add_patch(rect)
    ax.text(x_pos, 4.8, f's{i+1}', ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
    # 矢印: Embedding→LSTM
    ax.annotate('', xy=(x_pos, 4.2), xytext=(x_pos, 3.3),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
    # FC層 + 出力
    rect = mpatches.FancyBboxPatch((x_pos - 0.6, 6.2), 1.2, 0.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor=out_color, alpha=0.7)
    ax.add_patch(rect)
    ax.text(x_pos, 6.6, 'FC', ha='center', va='center', fontsize=9, color='white')
    # 矢印: LSTM→FC
    ax.annotate('', xy=(x_pos, 6.2), xytext=(x_pos, 5.4),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
    # 出力ラベル
    ax.text(x_pos, 7.5, out, ha='center', va='center', fontsize=12,
            fontweight='bold', color=out_color)
    # 矢印: FC→出力
    ax.annotate('', xy=(x_pos, 7.2), xytext=(x_pos, 7.0),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
    # 矢印: LSTM→LSTM (横)
    if i < len(dec_inputs) - 1:
        ax.annotate('', xy=(x_pos + 1.2, 4.8), xytext=(x_pos + 0.6, 4.8),
                    arrowprops=dict(arrowstyle='->', lw=2, color=dec_color))

# 矢印: コンテキスト → 最初のDecoder
ax.annotate('', xy=(9.5 - 0.6, 4.8), xytext=(ctx_x + 0.8, 4.8),
            arrowprops=dict(arrowstyle='->', lw=2.5, color=ctx_color))

# ラベル
ax.text(3.5, 0.5, '入力系列（ソース）', ha='center', fontsize=12,
        style='italic', color='gray')
ax.text(12.5, 0.5, '入力（前のステップの出力）', ha='center', fontsize=12,
        style='italic', color='gray')
ax.text(12.5, 8.3, '出力（予測トークン）', ha='center', fontsize=12,
        style='italic', color=out_color)

plt.tight_layout()
plt.show()

print("Encoder → Context Vector → Decoder の情報フローを確認しましょう")

In [ ]:
# ============================================================
# Encoder の実装
# 入力系列をコンテキストベクトルに変換するモジュール
# ============================================================

class Encoder(nn.Module):
    """Seq2SeqのEncoder（符号化器）
    
    入力系列をEmbedding → LSTM で処理し、
    最終隠れ状態（コンテキストベクトル）を返す。
    
    Args:
        input_dim:  入力語彙サイズ
        embed_dim:  埋め込み次元
        hidden_dim: LSTM隠れ層の次元
        n_layers:   LSTMの層数
    """
    def __init__(self, input_dim, embed_dim, hidden_dim, n_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # 単語をベクトルに変換する埋め込み層
        self.embedding = nn.Embedding(input_dim, embed_dim)
        
        # LSTMで系列を処理
        # batch_first=True: 入力を (batch, seq_len, features) 形式に
        self.rnn = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
    
    def forward(self, src):
        """順伝播
        
        Args:
            src: 入力系列 (batch_size, src_len)
        
        Returns:
            hidden: 最終隠れ状態 (n_layers, batch_size, hidden_dim)
            cell:   最終セル状態 (n_layers, batch_size, hidden_dim)
        """
        # src: (batch_size, src_len)
        embedded = self.embedding(src)
        # embedded: (batch_size, src_len, embed_dim)
        
        outputs, (hidden, cell) = self.rnn(embedded)
        # hidden: (n_layers, batch_size, hidden_dim) ← これがコンテキストベクトル
        # cell:   (n_layers, batch_size, hidden_dim)
        
        return hidden, cell

# 動作確認
enc_test = Encoder(input_dim=13, embed_dim=16, hidden_dim=32, n_layers=1)
dummy_src = torch.randint(0, 10, (4, 8))  # batch=4, seq_len=8
h, c = enc_test(dummy_src)
print(f"入力形状:           {dummy_src.shape}")
print(f"隠れ状態（context）: {h.shape}")
print(f"セル状態:           {c.shape}")
print("\nEncoderの動作確認完了")

In [ ]:
# ============================================================
# Decoder の実装
# コンテキストベクトルから出力系列を1ステップずつ生成するモジュール
# ============================================================

class Decoder(nn.Module):
    """Seq2SeqのDecoder（復号化器）
    
    1トークンずつ処理し、次のトークンを予測する。
    Encoderから受け取った隠れ状態（コンテキストベクトル）を
    初期状態として使用する。
    
    Args:
        output_dim: 出力語彙サイズ
        embed_dim:  埋め込み次元
        hidden_dim: LSTM隠れ層の次元
        n_layers:   LSTMの層数
    """
    def __init__(self, output_dim, embed_dim, hidden_dim, n_layers=1):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # 出力トークンをベクトルに変換する埋め込み層
        self.embedding = nn.Embedding(output_dim, embed_dim)
        
        # LSTMで1ステップ処理
        self.rnn = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        
        # 隠れ状態から語彙への線形変換
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, input_token, hidden, cell):
        """1ステップの順伝播
        
        Args:
            input_token: 現在の入力トークン (batch_size,)
            hidden: 前ステップの隠れ状態 (n_layers, batch_size, hidden_dim)
            cell:   前ステップのセル状態 (n_layers, batch_size, hidden_dim)
        
        Returns:
            prediction: 次トークンの予測 (batch_size, output_dim)
            hidden: 更新された隠れ状態
            cell:   更新されたセル状態
        """
        # input_token: (batch_size,) → (batch_size, 1) にunsqueeze
        embedded = self.embedding(input_token.unsqueeze(1))
        # embedded: (batch_size, 1, embed_dim)
        
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output: (batch_size, 1, hidden_dim)
        
        # squeeze(1)で系列長次元を除去して線形層に通す
        prediction = self.fc(output.squeeze(1))
        # prediction: (batch_size, output_dim)
        
        return prediction, hidden, cell

# 動作確認
dec_test = Decoder(output_dim=13, embed_dim=16, hidden_dim=32, n_layers=1)
dummy_token = torch.randint(0, 10, (4,))  # batch=4, 1トークン
pred, h_new, c_new = dec_test(dummy_token, h, c)
print(f"入力トークン形状:   {dummy_token.shape}")
print(f"予測出力形状:       {pred.shape}")
print(f"更新された隠れ状態: {h_new.shape}")
print("\nDecoderの動作確認完了")

In [ ]:
# ============================================================
# Seq2Seq モデル全体の実装
# EncoderとDecoderを組み合わせ、Teacher Forcingを含む
# ============================================================

class Seq2Seq(nn.Module):
    """Seq2Seqモデル（Encoder + Decoder）
    
    Encoderで入力系列をコンテキストベクトルに変換し、
    Decoderで出力系列を1トークンずつ生成する。
    訓練時はTeacher Forcingを使用可能。
    
    Args:
        encoder: Encoderモジュール
        decoder: Decoderモジュール
        device:  計算デバイス (cpu/cuda)
    """
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """順伝播
        
        Args:
            src: 入力系列 (batch_size, src_len)
            trg: ターゲット系列 (batch_size, trg_len)
            teacher_forcing_ratio: Teacher Forcingの確率
                1.0 = 常に正解を使用
                0.0 = 常にモデルの予測を使用
        
        Returns:
            outputs: 各ステップの予測 (batch_size, trg_len, trg_vocab_size)
        """
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        
        # 出力を格納するテンソル
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        # Encoderで入力系列をコンテキストベクトルに変換
        hidden, cell = self.encoder(src)
        
        # Decoderの最初の入力は <SOS> トークン
        input_token = trg[:, 0]  # <SOS> token
        
        # 1トークンずつ生成
        for t in range(1, trg_len):
            # Decoderで1ステップ予測
            prediction, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t] = prediction
            
            # Teacher Forcing: 確率的に正解 or 予測を次の入力にする
            if np.random.random() < teacher_forcing_ratio:
                # 正解トークンを次の入力にする（Teacher Forcing）
                input_token = trg[:, t]
            else:
                # モデル自身の予測を次の入力にする（自己回帰）
                input_token = prediction.argmax(dim=1)
        
        return outputs

# 動作確認
enc = Encoder(input_dim=13, embed_dim=16, hidden_dim=32)
dec = Decoder(output_dim=13, embed_dim=16, hidden_dim=32)
model_test = Seq2Seq(enc, dec, device)

dummy_src = torch.randint(0, 10, (4, 8))
dummy_trg = torch.randint(0, 10, (4, 9))
out = model_test(dummy_src, dummy_trg, teacher_forcing_ratio=0.5)
print(f"入力形状:   {dummy_src.shape}")
print(f"ターゲット: {dummy_trg.shape}")
print(f"出力形状:   {out.shape}")
print("\nSeq2Seqモデルの動作確認完了")

# ============================================================
# 4. 数値系列逆順タスク
# ============================================================

## 4. 数値系列逆順タスク

### 🎯 タスクの概要

Seq2Seqの動作を理解するために、**数値系列を逆順にするタスク** を使います。

```
入力:  [3, 1, 4, 1, 5] → 出力: [5, 1, 4, 1, 3]
入力:  [2, 7, 1]       → 出力: [1, 7, 2]
```

### なぜこのタスクか？

1. **トークナイザー不要**: 数字をそのままトークンとして使える
2. **翻訳の本質を捉えている**: 入力系列全体を「理解」して出力する必要がある
3. **正解の検証が容易**: 逆順になっているかどうかすぐ分かる
4. **難易度調整が可能**: 系列長を変えることで難易度を調整できる

### 語彙（Vocabulary）

| トークン | ID | 用途 |
|----------|-----|------|
| 0-9 | 0-9 | 数字 |
| SOS | 10 | 系列開始 |
| EOS | 11 | 系列終了 |
| PAD | 12 | パディング |

In [ ]:
# ============================================================
# データ生成関数
# 数値系列とその逆順のペアを生成します
# ============================================================

# 特殊トークンの定義
SOS = 10  # Start of Sequence
EOS = 11  # End of Sequence
PAD = 12  # Padding
VOCAB_SIZE = 13  # 0-9 + SOS + EOS + PAD

def generate_reverse_data(n_samples=5000, min_len=3, max_len=8):
    """数値系列逆順タスクのデータを生成
    
    Args:
        n_samples: サンプル数
        min_len:   系列の最小長
        max_len:   系列の最大長
    
    Returns:
        src_tensor: 入力系列 (n_samples, max_len + 1)  # +1 for EOS
        trg_tensor: 出力系列 (n_samples, max_len + 2)  # +2 for SOS, EOS
    """
    src_seqs, trg_seqs = [], []
    
    for _ in range(n_samples):
        # ランダムな長さの数字系列を生成
        length = np.random.randint(min_len, max_len + 1)
        seq = np.random.randint(0, 10, length).tolist()
        
        # ソース: 数字 + EOS + PAD...
        src = seq + [EOS] + [PAD] * (max_len - length)
        
        # ターゲット: SOS + 逆順の数字 + EOS + PAD...
        trg = [SOS] + seq[::-1] + [EOS] + [PAD] * (max_len - length)
        
        src_seqs.append(src)
        trg_seqs.append(trg)
    
    return torch.tensor(src_seqs, dtype=torch.long), torch.tensor(trg_seqs, dtype=torch.long)

# データ生成
np.random.seed(42)
src_data, trg_data = generate_reverse_data(n_samples=5000)

# データの確認
print("="*60)
print("生成されたデータの例")
print("="*60)
for i in range(5):
    src = src_data[i].tolist()
    trg = trg_data[i].tolist()
    # PAD以外の部分を表示
    src_clean = [x for x in src if x != PAD]
    trg_clean = [x for x in trg if x != PAD]
    print(f"  入力: {src_clean}")
    print(f"  出力: {trg_clean}")
    print()

print(f"データ形状:")
print(f"  ソース: {src_data.shape}")
print(f"  ターゲット: {trg_data.shape}")
print(f"  語彙サイズ: {VOCAB_SIZE}")

In [ ]:
# ============================================================
# 訓練ループ
# Seq2Seqモデルを数値系列逆順タスクで訓練します
# ============================================================

def train_seq2seq(hidden_dim=64, embed_dim=32, n_epochs=50,
                  lr=0.001, teacher_forcing_ratio=0.5,
                  src_data=src_data, trg_data=trg_data,
                  batch_size=128, verbose=True):
    """Seq2Seqモデルの訓練
    
    Args:
        hidden_dim: 隠れ層の次元
        embed_dim:  埋め込み次元
        n_epochs:   エポック数
        lr:         学習率
        teacher_forcing_ratio: Teacher Forcingの確率
        src_data:   ソースデータ
        trg_data:   ターゲットデータ
        batch_size: バッチサイズ
        verbose:    進捗を表示するか
    
    Returns:
        model:  訓練済みモデル
        losses: 各エポックの損失リスト
    """
    # シード固定
    torch.manual_seed(42)
    np.random.seed(42)
    
    # モデルの構築
    encoder = Encoder(VOCAB_SIZE, embed_dim, hidden_dim)
    decoder = Decoder(VOCAB_SIZE, embed_dim, hidden_dim)
    model = Seq2Seq(encoder, decoder, device).to(device)
    
    # 最適化とロス
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # PADトークンを無視してロスを計算
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    
    # 訓練データの分割
    n_train = int(len(src_data) * 0.8)
    train_src, train_trg = src_data[:n_train], trg_data[:n_train]
    
    losses = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        
        # ミニバッチ学習
        indices = np.random.permutation(n_train)
        for start in range(0, n_train, batch_size):
            idx = indices[start:start + batch_size]
            batch_src = train_src[idx].to(device)
            batch_trg = train_trg[idx].to(device)
            
            optimizer.zero_grad()
            
            # 順伝播
            output = model(batch_src, batch_trg, teacher_forcing_ratio)
            
            # ロス計算
            # output: (batch, trg_len, vocab) → (batch * trg_len, vocab)
            # trg:    (batch, trg_len) → (batch * trg_len)
            output_flat = output[:, 1:].contiguous().view(-1, VOCAB_SIZE)
            trg_flat = batch_trg[:, 1:].contiguous().view(-1)
            
            loss = criterion(output_flat, trg_flat)
            
            # 逆伝播
            loss.backward()
            # 勾配クリッピング（勾配爆発の防止）
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs}: Loss = {avg_loss:.4f}")
    
    return model, losses

# モデルの訓練
print("="*60)
print("Seq2Seqモデルの訓練開始")
print("="*60)
model, losses = train_seq2seq(hidden_dim=64, embed_dim=32, n_epochs=50,
                               teacher_forcing_ratio=0.5)
print(f"\n最終損失: {losses[-1]:.4f}")

In [ ]:
# ============================================================
# 訓練損失の可視化
# エポックごとの損失変化を確認します
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 損失曲線
axes[0].plot(range(1, len(losses) + 1), losses, linewidth=2, color='steelblue')
axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('損失（CrossEntropy）', fontsize=12)
axes[0].set_title('訓練損失の推移', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 右: 対数スケール
axes[1].plot(range(1, len(losses) + 1), losses, linewidth=2, color='coral')
axes[1].set_xlabel('エポック', fontsize=12)
axes[1].set_ylabel('損失（対数スケール）', fontsize=12)
axes[1].set_title('訓練損失の推移（対数スケール）', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"初期損失: {losses[0]:.4f} → 最終損失: {losses[-1]:.4f}")
print(f"損失の減少率: {(1 - losses[-1] / losses[0]) * 100:.1f}%")

In [ ]:
# ============================================================
# 評価：モデルの予測を確認
# 訓練済みモデルで系列逆順タスクの結果を見ます
# ============================================================

def evaluate(model, src_tensor, max_len=12):
    """モデルの推論（Teacher Forcingなし）
    
    Args:
        model:      訓練済みSeq2Seqモデル
        src_tensor: 入力系列 (1, src_len) or (src_len,)
        max_len:    最大出力長
    
    Returns:
        predicted: 予測トークンのリスト
    """
    model.eval()
    
    if src_tensor.dim() == 1:
        src_tensor = src_tensor.unsqueeze(0)
    
    with torch.no_grad():
        # Encoderでコンテキストベクトルを取得
        hidden, cell = model.encoder(src_tensor)
        
        # Decoderで1トークンずつ生成
        input_token = torch.tensor([SOS]).to(device)
        predicted = []
        
        for _ in range(max_len):
            prediction, hidden, cell = model.decoder(input_token, hidden, cell)
            # 最も確率の高いトークンを選択
            top1 = prediction.argmax(dim=1)
            predicted.append(top1.item())
            
            # EOSが出たら終了
            if top1.item() == EOS:
                break
            
            input_token = top1
    
    return predicted

def compute_accuracy(model, src_data, trg_data, n_samples=500):
    """系列単位の正解率を計算"""
    correct = 0
    total = min(n_samples, len(src_data))
    
    for i in range(total):
        predicted = evaluate(model, src_data[i])
        # ターゲット: SOS以降、PAD/EOSを除去
        target = [t.item() for t in trg_data[i][1:] if t.item() not in (PAD, EOS)]
        # 予測からEOSを除去
        pred_clean = [p for p in predicted if p not in (PAD, EOS)]
        
        if pred_clean == target:
            correct += 1
    
    return correct / total

# テストデータで評価
n_train = int(len(src_data) * 0.8)
test_src, test_trg = src_data[n_train:], trg_data[n_train:]

print("="*60)
print("モデルの予測結果（テストデータ）")
print("="*60)

for i in range(10):
    src = test_src[i]
    trg = test_trg[i]
    
    # 入力（PAD/EOSを除く）
    src_clean = [s.item() for s in src if s.item() not in (PAD, EOS)]
    # 正解（SOS/PAD/EOSを除く）
    trg_clean = [t.item() for t in trg if t.item() not in (SOS, PAD, EOS)]
    # 予測
    predicted = evaluate(model, src)
    pred_clean = [p for p in predicted if p not in (PAD, EOS)]
    
    match = '✅' if pred_clean == trg_clean else '❌'
    print(f"  入力: {src_clean} → 予測: {pred_clean} (正解: {trg_clean}) {match}")

# 全体の正解率
acc = compute_accuracy(model, test_src, test_trg, n_samples=500)
print(f"\nテスト正解率（系列完全一致）: {acc:.1%}")

# ============================================================
# 5. Teacher Forcing
# ============================================================

## 5. Teacher Forcing

### 🤔 Teacher Forcingとは？

Decoderは各ステップで「次のトークン」を予測しますが、
その入力として何を使うかに2つの方法があります：

#### 方法1: Teacher Forcing（教師あり）
```
ステップ1: 入力=<SOS>      → 予測=5  （正解=5）
ステップ2: 入力=5（正解）  → 予測=1  （正解=1）
ステップ3: 入力=1（正解）  → 予測=4  （正解=4）
```
- **正解トークン** を次の入力にする
- 訓練が速く安定する
- 推論時とのギャップ（exposure bias）がある

#### 方法2: 自己回帰（Teacher Forcingなし）
```
ステップ1: 入力=<SOS>      → 予測=5  （正解=5）
ステップ2: 入力=5（予測）  → 予測=3  （正解=1）← 間違い！
ステップ3: 入力=3（予測）  → 予測=7  （正解=4）← エラーが伝播
```
- **モデル自身の予測** を次の入力にする
- 推論時と同じ条件で訓練できる
- 初期はエラーが蓄積しやすく、訓練が不安定

#### 方法3: Scheduled Sampling（ハイブリッド）
- Teacher Forcing比率を徐々に下げていく
- 例: 最初は1.0 → 訓練後半で0.5 → 最後は0.0

### ⚠️ 重要な注意

**推論（テスト）時は必ずTeacher Forcing = 0.0にすること！**

推論時は正解が分からないため、モデル自身の予測を使うしかありません。

In [ ]:
# ============================================================
# Teacher Forcing比率の効果を比較
# 異なるTF比率で訓練し、学習曲線と精度を比較します
# ============================================================

tf_ratios = [0.0, 0.5, 1.0]
tf_results = {}

print("="*60)
print("Teacher Forcing比率の比較実験")
print("="*60)

for tf_ratio in tf_ratios:
    print(f"\nTeacher Forcing = {tf_ratio}:")
    model_tf, losses_tf = train_seq2seq(
        hidden_dim=64, embed_dim=32, n_epochs=50,
        teacher_forcing_ratio=tf_ratio,
        verbose=True
    )
    
    # テストデータで精度を計算（TF=0.0で評価）
    acc = compute_accuracy(model_tf, test_src, test_trg, n_samples=500)
    tf_results[tf_ratio] = {'losses': losses_tf, 'accuracy': acc, 'model': model_tf}
    print(f"  テスト正解率: {acc:.1%}")

print("\n" + "="*60)
print("比較結果サマリー")
print("="*60)
for tf_ratio in tf_ratios:
    acc = tf_results[tf_ratio]['accuracy']
    final_loss = tf_results[tf_ratio]['losses'][-1]
    print(f"  TF={tf_ratio}: 最終損失={final_loss:.4f}, テスト正解率={acc:.1%}")

In [ ]:
# ============================================================
# Teacher Forcing比較の可視化
# 学習曲線と最終精度をグラフにします
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#f39c12', '#2ecc71']
labels = ['TF=0.0（自己回帰のみ）', 'TF=0.5（ハイブリッド）', 'TF=1.0（教師あり）']

# 左: 学習曲線
for i, tf_ratio in enumerate(tf_ratios):
    losses_tf = tf_results[tf_ratio]['losses']
    axes[0].plot(range(1, len(losses_tf) + 1), losses_tf,
                 linewidth=2, color=colors[i], label=labels[i])

axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('損失', fontsize=12)
axes[0].set_title('Teacher Forcing比率と学習曲線', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 右: 精度比較（棒グラフ）
accuracies = [tf_results[tf]['accuracy'] for tf in tf_ratios]
bars = axes[1].bar([f'TF={r}' for r in tf_ratios], accuracies,
                    color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

# 棒グラフの上に数値を表示
for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{acc:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

axes[1].set_xlabel('Teacher Forcing比率', fontsize=12)
axes[1].set_ylabel('テスト正解率', fontsize=12)
axes[1].set_title('Teacher Forcing比率とテスト精度', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Teacher Forcing比率0.5が訓練安定性とテスト性能のバランスが良いことが多いです")

# ============================================================
# 6. コンテキストベクトルのボトルネック
# ============================================================

## 6. コンテキストベクトルのボトルネック

### 🤔 ボトルネック問題とは？

Seq2Seqの最大の弱点は、**入力系列のすべての情報を1つの固定長ベクトル**（コンテキストベクトル）に
圧縮しなければならないことです。

```
短い入力:  [3, 1, 4] ──→ [context: 64次元] ──→ 十分な情報
長い入力:  [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 8, 9, 7, 9, 3, 2, 3, 8, 4, 6]
                     ──→ [context: 64次元] ──→ 情報が足りない！
```

### 予想される振る舞い

- **系列長が長い** ほど精度が下がる（多くの情報を圧縮する必要がある）
- **隠れ次元が大きい** ほど精度が向上する（圧縮に使えるベクトルが大きい）
- 系列長と隠れ次元の関係が性能に影響する

### 実験: 隠れ次元 × 系列長

隠れ次元と系列長を変えて、ボトルネックの影響を定量的に調べましょう。

**この問題を解決するのが、次のNotebook 126で学ぶ「Attention（注意機構）」です。**

In [ ]:
# ============================================================
# ボトルネック実験
# 隠れ次元と系列長を変えて精度を測定します
# ============================================================

hidden_dims = [16, 32, 64, 128]
seq_lengths = [5, 10, 15, 20]

# 結果を格納する2次元配列
accuracy_matrix = np.zeros((len(hidden_dims), len(seq_lengths)))

print("="*60)
print("ボトルネック実験: 隠れ次元 × 系列長")
print("="*60)

for i, h_dim in enumerate(hidden_dims):
    for j, seq_len in enumerate(seq_lengths):
        print(f"  hidden_dim={h_dim:3d}, seq_len={seq_len:2d} ... ", end='')
        
        # この条件のデータを生成
        np.random.seed(42)
        src_exp, trg_exp = generate_reverse_data(
            n_samples=3000, min_len=seq_len, max_len=seq_len
        )
        
        # 訓練（短めのエポック数で実験）
        model_exp, _ = train_seq2seq(
            hidden_dim=h_dim, embed_dim=32, n_epochs=40,
            teacher_forcing_ratio=0.5,
            src_data=src_exp, trg_data=trg_exp,
            verbose=False
        )
        
        # テストデータで評価
        n_train_exp = int(len(src_exp) * 0.8)
        test_src_exp = src_exp[n_train_exp:]
        test_trg_exp = trg_exp[n_train_exp:]
        acc = compute_accuracy(model_exp, test_src_exp, test_trg_exp, n_samples=200)
        accuracy_matrix[i, j] = acc
        
        print(f"精度={acc:.1%}")

print("\n実験完了！")

In [ ]:
# ============================================================
# ボトルネック実験の可視化
# ヒートマップとラインプロットで結果を表示します
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- 左: ヒートマップ ---
im = axes[0].imshow(accuracy_matrix, cmap='RdYlGn', aspect='auto',
                      vmin=0, vmax=1)

# 軸ラベル
axes[0].set_xticks(range(len(seq_lengths)))
axes[0].set_xticklabels(seq_lengths, fontsize=11)
axes[0].set_yticks(range(len(hidden_dims)))
axes[0].set_yticklabels(hidden_dims, fontsize=11)
axes[0].set_xlabel('系列長', fontsize=12)
axes[0].set_ylabel('隠れ次元', fontsize=12)
axes[0].set_title('ボトルネック: 隠れ次元 × 系列長', fontsize=14, fontweight='bold')

# 各セルに数値を表示
for i in range(len(hidden_dims)):
    for j in range(len(seq_lengths)):
        text_color = 'white' if accuracy_matrix[i, j] < 0.5 else 'black'
        axes[0].text(j, i, f'{accuracy_matrix[i, j]:.0%}',
                     ha='center', va='center', fontsize=12,
                     fontweight='bold', color=text_color)

plt.colorbar(im, ax=axes[0], label='正解率')

# --- 右: ラインプロット ---
colors_line = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
for i, h_dim in enumerate(hidden_dims):
    axes[1].plot(seq_lengths, accuracy_matrix[i], marker='o',
                 linewidth=2, markersize=8, color=colors_line[i],
                 label=f'hidden_dim={h_dim}')

axes[1].set_xlabel('系列長', fontsize=12)
axes[1].set_ylabel('テスト正解率', fontsize=12)
axes[1].set_title('系列長による精度低下', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("="*60)
print("分析結果")
print("="*60)
print("1. 系列長が長くなるほど精度が低下する → ボトルネックの証拠")
print("2. 隠れ次元を大きくすると改善するが、限界がある")
print("3. この問題を根本的に解決するのが Attention 機構")

# ============================================================
# 7. まとめ
# ============================================================

## 7. まとめ

### 🎯 このノートブックで学んだこと

**Seq2Seqアーキテクチャ**
- ✓ 可変長入力 → 可変長出力を実現するEncoder-Decoder構造
- ✓ Encoderが入力系列をコンテキストベクトルに圧縮
- ✓ Decoderがコンテキストベクトルから出力系列を生成

**Teacher Forcing**
- ✓ 訓練時に正解トークンをDecoderの入力に使う技法
- ✓ 訓練の安定化と高速化に寄与
- ✓ 推論時は必ずTF=0.0にする必要がある

**ボトルネック問題**
- ✓ コンテキストベクトルの固定長がボトルネックになる
- ✓ 系列が長くなるほど情報損失が増加する
- ✓ Attention機構（次回）がこの問題を解決する

### 📊 Seq2Seqチートシート

| 項目 | 内容 |
|------|------|
| **アーキテクチャ** | Encoder (LSTM) + Decoder (LSTM) |
| **入力** | 可変長トークン系列 + EOS |
| **出力** | SOS → 可変長トークン系列 → EOS |
| **情報の橋渡し** | コンテキストベクトル（Encoderの最終隠れ状態）|
| **訓練テクニック** | Teacher Forcing (ratio 0.5が一般的) |
| **損失関数** | CrossEntropyLoss (PADを無視) |
| **弱点** | ボトルネック（長い系列で性能低下）|
| **解決策** | Attention機構（Notebook 126）|

### ⚠️ よくある間違い

❌ **SOS/EOSトークンを忘れる**  
✅ Decoderの入力は必ず`<SOS>`から始め、出力は`<EOS>`で終わるようにする

❌ **推論時にTeacher Forcingを使う**  
✅ 推論時は正解が不明なので、必ず`teacher_forcing_ratio=0.0`にする

❌ **PADトークンでロスを計算する**  
✅ `nn.CrossEntropyLoss(ignore_index=PAD)`でPADを除外する

❌ **勾配クリッピングを忘れる**  
✅ RNN系は勾配爆発しやすいので、`clip_grad_norm_`を使う

### ✅ 学習チェックリスト

- [ ] Encoder-Decoderの情報フローを説明できる
- [ ] コンテキストベクトルの役割を理解している
- [ ] Teacher Forcingの仕組みと注意点を理解している
- [ ] PyTorchでSeq2Seqを実装できる
- [ ] ボトルネック問題を説明できる

---

**次のステップ**: Notebook 126で、ボトルネック問題を解決する **Attention（注意機構）** を学びます！

# ============================================================
# 8. 自己評価クイズ
# ============================================================

## 🎓 自己評価クイズ

学習内容を確認しましょう！すぐに答えを見ずに、まず自分で考えてみてください。

### Q1: コンテキストベクトルとは何ですか？その役割は？

<details>
<summary>💡 答えを見る</summary>

**答え**: コンテキストベクトルは、Encoderの最終隠れ状態（LSTMの場合はhiddenとcellの両方）です。

入力系列のすべての情報を固定長のベクトルに圧縮したものであり、
Decoderの初期状態として渡されます。Decoderはこのベクトルだけを頼りに
出力系列を生成します。

例: "I love cats" → [0.23, -0.15, 0.87, ...] (64次元ベクトル)

</details>

---

### Q2: なぜTeacher Forcingが必要なのですか？

<details>
<summary>💡 答えを見る</summary>

**答え**: Teacher Forcingがないと、Decoderの誤った予測が次のステップの入力になり、
エラーが連鎖的に蓄積します（exposure bias）。

特に訓練初期はモデルの予測がほぼランダムなため、
Teacher Forcingなしでは意味のある勾配が得られず、学習が非常に遅くなります。

Teacher Forcing比率0.5程度が、訓練の安定性と推論時の性能のバランスが良いとされています。

</details>

---

### Q3: ボトルネック問題とは何ですか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 入力系列の長さに関わらず、Encoderはすべての情報を固定長のコンテキストベクトルに
圧縮しなければなりません。

入力が長くなるほど、1つのベクトルに収める情報量が増え、情報損失が大きくなります。
これがボトルネック問題です。

実験でも示したように、系列長が長くなると精度が低下し、
隠れ次元を大きくしても限界があります。

この問題を解決するのがAttention機構で、Decoderが入力系列の各位置を
直接参照できるようにします。

</details>

---

### Q4: なぜ推論時にTeacher Forcingを使えないのですか？

<details>
<summary>💡 答えを見る</summary>

**答え**: Teacher Forcingは「正解トークン」をDecoderの次の入力として使う技法です。

しかし推論時（実際の翻訳・要約など）では、正解が手元にありません。
Decoderは自分自身の予測を頼りに、1トークンずつ自己回帰的に生成するしかありません。

したがって、推論時は `teacher_forcing_ratio=0.0` に設定し、
モデル自身の予測を次の入力として使います。

</details>

## 📚 参考文献

1. **Sutskever, I., Vinyals, O., & Le, Q. V.** (2014).  
   *Sequence to Sequence Learning with Neural Networks.*  
   Advances in Neural Information Processing Systems (NeurIPS).  
   — Seq2Seqの原論文。LSTMベースのEncoder-Decoderで機械翻訳を実現。

2. **Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., & Bengio, Y.** (2014).  
   *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation.*  
   Proceedings of EMNLP.  
   — GRUを用いたEncoder-Decoderモデル。Seq2Seqのもう1つの原論文。

---

**次のステップ**: Notebook 126で **Attention機構** を学び、ボトルネック問題を解決しましょう！